# 03 - Forecasting V2 (True t+1 Forecast)

Inputs:
- `data/splits/train_v2.csv`
- `data/splits/test_v2.csv`
- `outputs/models/feature_columns_v2.json`

Target in splits is `AQI` at `target_timestamp` (t+1 relative to feature timestamp).
This notebook compares: Naive baseline, Linear Regression, Random Forest, XGBoost.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')

TRAIN_PATH = Path('data/splits/train_v2.csv')
TEST_PATH = Path('data/splits/test_v2.csv')
FEATURES_PATH = Path('outputs/models/feature_columns_v2.json')

REPORTS_DIR = Path('outputs/reports')
MODELS_DIR = Path('outputs/models')
FIG_DIR = Path('outputs/figures')
for d in [REPORTS_DIR, MODELS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)


## Load Data And Feature Schema

In [ ]:
if not TRAIN_PATH.exists() or not TEST_PATH.exists() or not FEATURES_PATH.exists():
    raise FileNotFoundError('Missing v2 split/features artifacts. Run 02_preprocessing_v2 first.')

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
feature_columns = json.loads(FEATURES_PATH.read_text(encoding='utf-8'))

required_train = ['Timestamp', 'target_timestamp', 'City', 'AQI'] + feature_columns
required_test = ['Timestamp', 'target_timestamp', 'City', 'AQI'] + feature_columns
missing_train = [c for c in required_train if c not in train_df.columns]
missing_test = [c for c in required_test if c not in test_df.columns]
if missing_train:
    raise ValueError(f'Missing columns in train_v2: {missing_train}')
if missing_test:
    raise ValueError(f'Missing columns in test_v2: {missing_test}')

train_df['Timestamp'] = pd.to_datetime(train_df['Timestamp'])
test_df['Timestamp'] = pd.to_datetime(test_df['Timestamp'])
train_df['target_timestamp'] = pd.to_datetime(train_df['target_timestamp'])
test_df['target_timestamp'] = pd.to_datetime(test_df['target_timestamp'])

X_train = train_df[feature_columns].astype(float)
y_train = train_df['AQI'].astype(float)
X_test = test_df[feature_columns].astype(float)
y_test = test_df['AQI'].astype(float)

# Impute remaining feature NaNs using train medians only (no test fitting).
train_medians = X_train.median(numeric_only=True)
train_nan_before = int(X_train.isna().sum().sum())
test_nan_before = int(X_test.isna().sum().sum())
X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)
print(f'Filled NaNs using train medians -> train: {train_nan_before}, test: {test_nan_before}')

assert X_train.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0
assert y_train.isna().sum() == 0
assert y_test.isna().sum() == 0

print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Feature count:', len(feature_columns))


## Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100.0
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
preds = {}


## Naive Baseline (Correct Unit, True Forecast)

For each city and test target date `d`, predict AQI with last known AQI at date `d-1`.
This is computed in original AQI units from combined train+test target history.

In [ ]:
hist = pd.concat([
    train_df[['City', 'target_timestamp', 'AQI']].assign(split='train'),
    test_df[['City', 'target_timestamp', 'AQI']].assign(split='test')
], ignore_index=True)

hist = hist.sort_values(['City', 'target_timestamp']).reset_index(drop=True)
hist['naive_pred'] = hist.groupby('City')['AQI'].shift(1)

test_naive = hist[hist['split'] == 'test'][['City', 'target_timestamp', 'naive_pred']].copy()
test_aligned = test_df[['City', 'target_timestamp', 'AQI']].merge(
    test_naive, on=['City', 'target_timestamp'], how='left'
)

if test_aligned['naive_pred'].isna().sum() > 0:
    fallback = train_df.sort_values(['City', 'target_timestamp']).groupby('City')['AQI'].tail(1)
    fallback_map = train_df.sort_values(['City', 'target_timestamp']).groupby('City')['AQI'].last().to_dict()
    test_aligned['naive_pred'] = test_aligned.apply(
        lambda r: fallback_map.get(r['City'], np.nan) if pd.isna(r['naive_pred']) else r['naive_pred'], axis=1
    )

y_pred_naive = test_aligned['naive_pred'].values
metrics_naive = compute_metrics(y_test.values, y_pred_naive)
results.append({'Model': 'Naive Baseline', **metrics_naive})
preds['Naive Baseline'] = y_pred_naive
metrics_naive


## Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
metrics_lr = compute_metrics(y_test, y_pred_lr)
results.append({'Model': 'Linear Regression', **metrics_lr})
preds['Linear Regression'] = y_pred_lr
metrics_lr


## Random Forest

In [ ]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
)
try:
    rf.fit(X_train, y_train)
except PermissionError:
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=1,
    )
    rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
metrics_rf = compute_metrics(y_test, y_pred_rf)
results.append({'Model': 'Random Forest', **metrics_rf})
preds['Random Forest'] = y_pred_rf
metrics_rf


## XGBoost

In [ ]:
xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1,
)
try:
    xgb.fit(X_train, y_train)
except PermissionError:
    xgb = XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        objective='reg:squarederror',
        n_jobs=1,
    )
    xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
metrics_xgb = compute_metrics(y_test, y_pred_xgb)
results.append({'Model': 'XGBoost', **metrics_xgb})
preds['XGBoost'] = y_pred_xgb
metrics_xgb


## Comparison, Selection, And Save Artifacts

Best model rule: MAE, then RMSE, then MAPE (all ascending).

In [ ]:
comparison = pd.DataFrame(results).sort_values(['MAE', 'RMSE', 'MAPE']).reset_index(drop=True)
best_model_name = comparison.iloc[0]['Model']
display(comparison)
print('Best model:', best_model_name)

comparison.to_csv(REPORTS_DIR / 'model_comparison_v2.csv', index=False)

model_map = {
    'Linear Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb,
}
if best_model_name == 'Naive Baseline':
    joblib.dump({'type': 'naive_baseline_v2'}, MODELS_DIR / 'best_model_v2.pkl')
else:
    joblib.dump(model_map[best_model_name], MODELS_DIR / 'best_model_v2.pkl')

(MODELS_DIR / 'best_model_name_v2.txt').write_text(best_model_name, encoding='utf-8')

print('Saved:', REPORTS_DIR / 'model_comparison_v2.csv')
print('Saved:', MODELS_DIR / 'best_model_v2.pkl')
print('Saved:', MODELS_DIR / 'best_model_name_v2.txt')


## Build Evaluation Frame For Plots

In [ ]:
y_pred_best = preds[best_model_name]
eval_df = test_df[['City', 'Timestamp', 'target_timestamp']].copy()
eval_df['actual'] = y_test.values
eval_df['predicted'] = y_pred_best
eval_df['residual'] = eval_df['actual'] - eval_df['predicted']
eval_df['abs_error'] = eval_df['residual'].abs()
eval_df['target_month'] = eval_df['target_timestamp'].dt.month
eval_df.head()


## Plots

In [ ]:
# 1) Predicted vs Actual (first 180 target days, date-aggregated)
daily = eval_df.groupby('target_timestamp', as_index=False)[['actual', 'predicted']].mean().sort_values('target_timestamp')
plot_180 = daily.head(180)

plt.figure(figsize=(14, 6))
plt.plot(plot_180['target_timestamp'], plot_180['actual'], label='Actual AQI(t+1)', linewidth=2)
plt.plot(plot_180['target_timestamp'], plot_180['predicted'], label='Predicted AQI(t+1)', linewidth=2)
plt.title(f'V2 Predicted vs Actual (first 180 target days) - {best_model_name}')
plt.xlabel('Target Date')
plt.ylabel('AQI')
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'predicted_vs_actual_v2.png', dpi=200)
plt.close()

# 2) Residual histogram
plt.figure(figsize=(12, 6))
sns.histplot(eval_df['residual'], bins=60, kde=True)
plt.title(f'V2 Residual Distribution - {best_model_name}')
plt.xlabel('Residual (Actual - Predicted)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIG_DIR / 'residuals_distribution_v2.png', dpi=200)
plt.close()

# 3) Absolute error by month (derived from target timestamp)
plt.figure(figsize=(12, 6))
sns.boxplot(data=eval_df, x='target_month', y='abs_error', color='steelblue')
plt.title(f'V2 Absolute Error by Target Month - {best_model_name}')
plt.xlabel('Target Month')
plt.ylabel('Absolute Error')
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_month_v2.png', dpi=200)
plt.close()

# 4) Absolute error by city
order = eval_df.groupby('City')['abs_error'].median().sort_values(ascending=False).index
plt.figure(figsize=(14, 6))
sns.boxplot(data=eval_df, x='City', y='abs_error', order=order)
plt.title(f'V2 Absolute Error by City - {best_model_name}')
plt.xlabel('City')
plt.ylabel('Absolute Error')
plt.xticks(rotation=25)
plt.tight_layout()
plt.savefig(FIG_DIR / 'error_by_city_v2.png', dpi=200)
plt.close()

print('Saved v2 plots in', FIG_DIR)
